# N17 · 量化 Calibration

**关联 lab**: L08.5

**学习目标**: 把 SmoothQuant、AWQ、FP8 calibration 的数学从 paper 公式变成可断言的 numpy 复现。理解激活异常值、scale 求解、per-channel vs per-tensor 在精度上的差异。

**No-GPU 可完成度**: 100%。

**对应 MiniInfra**:
- `mini_infra/vllm/quant/calibrator.py`
- `mini_infra/vllm/quant/awq_loader.py`
- `mini_infra/vllm/quant/fp8_kv.py`
- `mini_infra/sglang/quant/kv_int8.py`

**对应真实源码**: `github_repo/llm-compressor`、`vllm/.../quantization/`

## 1. 量化的本质

把 fp16/fp32 张量映射到低精度（int8 / int4 / fp8），用 scale (与可选 zero_point) 表示：

$$x_{\text{quant}} = \text{round}((x - z) / s), \quad x_{\text{dequant}} = x_{\text{quant}} \cdot s + z$$

三选一的 scale 策略：
- **per-tensor**: 整张 tensor 一个 scale（最简单，精度最差）
- **per-channel**: 每 output channel 一个 scale
- **per-token**: KV cache 中每 token 一个 scale

用一个有 outlier 的 toy tensor 看精度差：

In [ ]:
import sys, math
from pathlib import Path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import random
random.seed(0)

# 模拟激活：100 个 token，每 token 4 channel
# channel 2 有 outlier（典型激活分布）
tokens = []
for t in range(100):
    row = [random.gauss(0, 0.5), random.gauss(0, 0.5), random.gauss(0, 5.0), random.gauss(0, 0.5)]
    if random.random() < 0.05:
        row[2] += random.gauss(0, 30.0)  # outlier 5% 概率
    tokens.append(row)

import statistics
for c in range(4):
    col = [t[c] for t in tokens]
    print(f'channel {c}: mean={statistics.mean(col):.2f}, std={statistics.stdev(col):.2f}, max_abs={max(abs(v) for v in col):.2f}')

In [ ]:
def quantize_int8(value, scale):
    return max(-127, min(127, round(value / scale)))

def dequantize_int8(q, scale):
    return q * scale

def errors(tokens, scales):
    """scales 可以是 scalar 或 list"""
    if isinstance(scales, (int, float)):
        get_scale = lambda c: scales
    else:
        get_scale = lambda c: scales[c]
    max_err = 0.0
    for token in tokens:
        for c, v in enumerate(token):
            s = get_scale(c)
            q = quantize_int8(v, s)
            d = dequantize_int8(q, s)
            max_err = max(max_err, abs(v - d))
    return max_err

# per-tensor: 整张 tensor max
max_abs = max(abs(v) for token in tokens for v in token)
scale_pt = max_abs / 127.0
err_pt = errors(tokens, scale_pt)

# per-channel: 每 channel 单独
scales_pc = [max(abs(t[c]) for t in tokens) / 127.0 for c in range(4)]
err_pc = errors(tokens, scales_pc)

print(f'per-tensor scale = {scale_pt:.4f}, max_err = {err_pt:.4f}')
print(f'per-channel scales = {[round(s, 4) for s in scales_pc]}')
print(f'per-channel max_err = {err_pc:.4f}')
print(f'  channel 0,1,3 (无 outlier) 在 per-channel 下精度提升 ~{scale_pt/scales_pc[0]:.1f}x')

**关键观察**: per-tensor 时 channel 2 的 outlier dominate scale，让 channel 0/1/3 的有效范围只用了 1/10。per-channel 直接救回。

这就是工业实践 99% 用 per-channel 的原因。

## 2. SmoothQuant 一阶平滑

SmoothQuant 想法: 把激活的 outlier "挪到" 权重那边。设激活 X，权重 W，等价变换：

$$Y = (X / s) \cdot (s \cdot W)$$

其中 s 是平滑因子。激活 X/s 后 outlier 减弱，权重 s·W 增大但仍可量化（权重通常分布平稳）。

$s$ 的选择：
$$s_j = \max(|X_j|)^{\alpha} \cdot \max(|W_j|)^{1-\alpha}$$

α=0: 不平滑（仅权重量化）；α=1: 全平滑（仅激活量化）；α=0.5 默认。

In [ ]:
from mini_infra.vllm.quant.calibrator import (
    channel_absmax, smoothquant_scales, awq_activation_order, calibration_summary
)

# 用 mini_infra 的实现
samples = tokens  # 复用上面 toy 数据
for alpha in [0.0, 0.3, 0.5, 0.7, 1.0]:
    scales = smoothquant_scales(samples, alpha=alpha)
    print(f'α={alpha}: scales = {[round(s, 4) for s in scales]}')

**观察**: α 越大，平滑越激进，channel 2 的 scale 越接近 max_abs。

实际效果：把 X/s 后再量化，比直接量化 X 误差小。

In [ ]:
# 验证 SmoothQuant 减小激活的 outlier
alpha = 0.5
scales = smoothquant_scales(samples, alpha=alpha)

smoothed = [[v / scales[c] for c, v in enumerate(token)] for token in samples]

for c in range(4):
    orig_max = max(abs(t[c]) for t in samples)
    smooth_max = max(abs(t[c]) for t in smoothed)
    print(f'channel {c}: orig max_abs={orig_max:.2f} → smoothed max_abs={smooth_max:.2f}  ({orig_max/smooth_max:.1f}x 缩小)')

**结论**: outlier channel（channel 2）的范围被缩小到与其他 channel 接近的量级。后续 per-channel quant 更容易。

代价：权重需要乘 s，但权重通常分布平稳，量化损失小。

## 3. AWQ：搜索权重 scale

AWQ (Activation-aware Weight Quantization) 思路: 对 4-bit 权重量化时，根据激活幅度选择保留的 "重要 channel"，对它们用更大 scale。

实现要点:
1. 在 calibration set 上算每 channel 激活 max
2. 把激活幅度排序，前 1% 通道视为 "salient"
3. 量化权重时，salient channel 单独 scale 保留精度

In [ ]:
order = awq_activation_order(samples)
absmax = channel_absmax(samples)
print('Channel ranking by activation absmax (descending):')
for rank, c in enumerate(order):
    print(f'  rank {rank}: channel {c} (absmax={absmax[c]:.2f})')
print(f'\nTop-1 salient channel: {order[0]} → 量化时单独保护')

## 4. FP8 vs INT8 数值精度

FP8 (e4m3) 范围 ±240, 4 位指数 + 3 位尾数；INT8 范围 ±127。

对小值 FP8 精度更高（小指数下尾数可表示精细差），对大值 INT8 线性量化精度更稳定。

In [ ]:
from mini_infra.vllm.quant.fp8_kv import fp8_scale, quantize_to_fp8, dequantize_from_fp8, fp8_summary

for max_abs in [0.1, 1.0, 10.0, 100.0]:
    values = [random.gauss(0, max_abs / 3) for _ in range(100)]
    
    # FP8
    s_fp8 = fp8_scale(max(abs(v) for v in values))
    q_fp8 = quantize_to_fp8(values, s_fp8)
    d_fp8 = dequantize_from_fp8(q_fp8, s_fp8)
    err_fp8 = max(abs(v - d) for v, d in zip(values, d_fp8))
    
    # INT8
    s_int8 = max(abs(v) for v in values) / 127.0
    q_int8 = [max(-127, min(127, round(v / s_int8))) for v in values]
    d_int8 = [q * s_int8 for q in q_int8]
    err_int8 = max(abs(v - d) for v, d in zip(values, d_int8))
    
    print(f'max_abs={max_abs:>6.1f}: FP8 max_err={err_fp8:.5f}, INT8 max_err={err_int8:.5f}')

**结论**: 范围小（< 1）时 FP8 精度优势明显；范围大时 INT8 与 FP8 相近。

实际应用：
- weights：分布通常 [-1, 1]，FP8 (e4m3) 优于 INT8
- activations：常有 outlier，FP8 (e5m2 范围大) 优于 INT8
- KV cache：常有 outlier 但范围中等，FP8 仍优

## 5. AWQ 显存账

用 awq_summary 看 AWQ w4a16 vs fp16 的显存差异：

In [ ]:
from mini_infra.vllm.quant.awq_loader import awq_summary, group_layout

for hidden in [4096, 8192, 14336]:
    s = awq_summary(hidden_size=hidden, group_size=128, weight_bits=4)
    print(f'hidden={hidden:>5}: '
          f'fp16 = {s["fp16_weight_gb"]:.2f} GB, '
          f'awq w4 = {s["packed_weight_gb"]:.2f} GB, '
          f'compression = {s["compression_ratio"]:.2f}x')

**注意**: compression < 4x（虽然 weight 4-bit 是 4x），因为还要存 scales (fp16 per-group)。group_size=128 时开销小（<5%），group_size=32 时开销大（~12%）。

## 6. 自检问题

1. per-tensor / per-channel / per-token 三种 scale 在显存与精度上的权衡？
2. SmoothQuant α=0.5 时，激活 outlier 被减弱多少？权重的代价是什么？
3. AWQ 与 SmoothQuant 各自适合什么量化目标（w4a16 vs w8a8）？
4. FP8 (e4m3) 与 INT8 在范围 [0.1, 1.0] 上谁精度更高？范围 [10, 100] 呢？
5. AWQ group_size 从 128 改成 32，显存与精度各怎么变？

## 7. 与 lab 对接

Lab 任务：在同模型同 workload 上跑 4 个 server（fp16 / fp8 / awq / kv-int8），对照 acc / latency / mem 矩阵。

**关键决策**：
- weights：AWQ (w4a16) 或 FP8 (e4m3)
- activations：fp16 (最稳) 或 FP8 (e5m2 速度快)
- KV cache：fp16 (acc 优先) 或 FP8 (e4m3, H100/H200) 或 INT8 (老硬件)

失败时按 ticket：
- AWQ load 报错 → `awq_load_fail_001`（group_size / packing）
- FP8 acc 大跌 → `fp8_calibration_002`（calibration / SmoothQuant）
- KV-int8 高并发漂移 → `kv_int8_acc_drift_003`（per-token scale）